<a href="https://colab.research.google.com/github/ngonhatanhly-NNA/AI-Training/blob/main/week1_data_preparation_fairness_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install datasets

from datasets import load_dataset
dataset = load_dataset("LabHC/bias_in_bios")
df = pd.DataFrame(dataset['train'])
df.head()

In [ ]:
print(df.info())
print(df.columns)
df.sample(3)

In [ ]:
df = df.drop_duplicates()
print(df.isnull().sum()) # Check for missing value
df = df.dropna(subset=["hard_text", "gender"])

In [ ]:
# Clean the dataset (remove space and non character)
df["hard_text"] = df["hard_text"].str.replace(r'[^a-zA-Z\s]', '', regex=True)

df["hard_text"] = df["hard_text"].str.lower()
df["hard_text"] = df["hard_text"].str.strip()
df["hard_text"].sample(3)

In [ ]:
# Analyze fairness in initial step (gender balance)
print(df["gender"].value_counts())

#Visualize
sns.countplot(x="gender", data=df)
plt.title("Gender Distribution")
plt.show()
#

In [ ]:
# Crosstab check gender statistic in career
gender_prof_cross = pd.crosstab(df['profession'], df['gender'])
gender_prof_ratio = pd.crosstab(df['profession'], df['gender'], normalize='index')
print(gender_prof_ratio.head())

import matplotlib.pyplot as plt

gender_prof_ratio.plot(kind='bar', stacked=True, figsize=(14, 6), colormap='Set2')
plt.title('Tỷ lệ Giới tính phân bố theo từng Nghề nghiệp')
plt.xlabel('Nghề nghiệp')
plt.ylabel('Tỷ lệ %')
plt.legend(title='Giới tính', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Train and test
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train, test = train_test_split(df, test_size=0.2, stratify=df["gender"], random_state=42)
print(train.shape, test.shape)

In [ ]:
# Save to csv
train.to_csv("biosbias_train_cleaned.csv", index=False)
test.to_csv("biosbias_test_clean", index=False)

WEEK 2: PIPLINE NLP and Scikit fudamental

In [ ]:
!pip install scikit-learn --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
X = df["hard_text"] # Text input
y = df["gender"] # Categorize
# Sensitive attr
sensitive_attr = df["gender"]

# Split train/test for repetition
X_train, X_test, y_train, y_test, gender_train, gender_test = train_test_split(X, y, sensitive_attr, test_size=0.2, stratify=y, random_state=42)
# TF-IDF vectorize (Tokenize and Vectorize) Evaluate term freq and inverse document Freq (tần số xuất hiên và độ phổ biến)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Using RegressionLogistic
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

# Prefiction
y_pred = clf.predict(X_test_tfidf)
# Evaluate
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix by Fairness Group')
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. Tạo một DataFrame chứa kết quả dự đoán trên tập test
results_df = pd.DataFrame({
    'True_Profession': y_test,
    'Predicted_Profession': y_pred,
    'Gender': gender_test
})

# 2. Tính Accuracy tổng thể
overall_accuracy = accuracy_score(results_df['True_Profession'], results_df['Predicted_Profession'])
print(f"Overall Accuracy: {overall_accuracy:.4f}")

# 3. Tính Subgroup Accuracy (Độ chính xác chia theo giới tính), xem tỷ lệ dự đoán đúng là bn, khi yêu cầu AI đoán
print("\n--- Accuracy by Gender ---")
subgroup_acc = results_df.groupby('Gender').apply(
    lambda x: accuracy_score(x['True_Profession'], x['Predicted_Profession'])
)
print(subgroup_acc)

# 4. Tính Accuracy Disparity (Khoảng cách thiên vị)
acc_diff = abs(subgroup_acc.iloc[0] - subgroup_acc.iloc[1])
print(f"\nAccuracy Disparity (Chênh lệch độ chính xác giữa các giới): {acc_diff:.4f}")